In [ ]:
import json
import os

# your credentials
kaggle_token = {"username":"your-kaggle-username","key":"your-api-key"}

# save to kaggle.json
os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_token, f)

os.chmod("/root/.kaggle/kaggle.json", 600)

In [ ]:
!kaggle competitions download -c house-prices-advanced-regression-techniques
!unzip house-prices-advanced-regression-techniques.zip

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
import os

In [ ]:
#Load train and test datasets
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(train.shape, test.shape)
train.head()


In [ ]:
train = train.drop(columns='Id' ,axis = 1)
test = test.drop(columns='Id' ,axis = 1)
train.head()

In [ ]:
# quick info of dataset
train.info()

In [ ]:
train.describe()

In [ ]:
# total n.o of missing values in each column
train.isnull().sum().sort_values(ascending=False).head(12)

In [ ]:
# perecentage of missing values
train.isnull().mean() * 100

In [ ]:
# missing values summary

# table
missing_counts = train.isnull().sum()
missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)
missing_pct = (missing_counts / len(train)) * 100
missing_df = pd.DataFrame({"missing_count": missing_counts, "missing_pct": missing_pct})

print("Columns with missing values:\n")
print(missing_df.head(15))

# plot
plt.figure(figsize=(10,5))
sns.barplot(x=missing_df.index, y=missing_df.missing_pct, color="steelblue")
plt.xticks(rotation=90)
plt.ylabel("% missing")
plt.title("Missing values per column (%)")
plt.show()

Note : Columns like PoolQC, Alley, and MiscFeature have >95% missing values, meaning they provide little value. I decided to drop them. For Fence (80% missing), I may keep it by treating “missing” as “no fence,” but dropping is also acceptable depending on downstream modeling needs.

In [ ]:
columns_to_drop = ['PoolQC' ,'Alley' ,'MiscFeature']
# training data
train = train.drop(columns=columns_to_drop , axis = 1)
# testing data
test = test.drop(columns=columns_to_drop , axis = 1)
print("Remaining cols after drop :" ,train.shape[1])

Note : I dropped PoolQC, Alley, and MiscFeature since they had >95% missing values and were unlikely to add predictive value.

In [ ]:
# Categorical features where NA means "None"
cat_fill_none = [
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'FireplaceQu', 'MasVnrType'
]

for col in cat_fill_none:
    train[col] = train[col].fillna("None")
    test[col]  = test[col].fillna("None")

# Numerical features where NA means 0 (not present)
num_fill_zero = [
    'GarageYrBlt', 'GarageArea', 'GarageCars',
    'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
    'BsmtFullBath', 'BsmtHalfBath',
    'MasVnrArea'
]

for col in num_fill_zero:
    train[col] = train[col].fillna(0)
    test[col]  = test[col].fillna(0)



Note :  I treated missing values in garage, basement, fireplace, and masonry features as structural absence. Categorical fields (e.g., GarageType, BsmtQual) were filled with "None", while numeric fields (e.g., MasVnrArea, BsmtFinSF1) were filled with 0. This preserves the true meaning of “not present.”

In [ ]:
train['LotFrontage'] =train.groupby('Neighborhood')['LotFrontage'].transform(lambda x : x.fillna(x.median()))
test['LotFrontage'] = test.groupby('Neighborhood')['LotFrontage'].transform(lambda x : x.fillna(x.median()))

Note : Since lot sizes are influenced by neighborhood planning, I imputed missing LotFrontage values with the median frontage of the respective Neighborhood. This preserves locality patterns and avoids bias from using a single global statistic.

In [ ]:
train['Fence']=  train['Fence'].fillna('None')
test['Fence']=  test['Fence'].fillna('None')

In [ ]:
# checking back again , if any missing values are left
print(train.isnull().sum().sort_values(ascending = False).head(10))
print(test.isnull().sum().sort_values(ascending = False).head(10))

In [ ]:
# fill Electrical in train
train['Electrical'] = train['Electrical'].fillna(train['Electrical'].mode()[0])

# Fill categorical vars in test with mode
for col in ['MSZoning', 'Utilities', 'Functional', 'Exterior1st',
            'Exterior2nd', 'KitchenQual', 'SaleType']:
    test[col] = test[col].fillna(test[col].mode()[0])

Note : I imputed the few remaining categorical missing values with their respective most frequent category (mode). This ensures minimal distortion, since the percentage of missingness was tiny (<0.5%).

In [ ]:
# checking back again , if any missing values are left
print(train.isnull().sum().sort_values(ascending = False).head(10))
print(test.isnull().sum().sort_values(ascending = False).head(10))

# UNIVARIATE ANALYSIS

In [ ]:
from scipy.stats import skew
plt.figure(figsize=(12,5))

# Histogram on log scale
plt.subplot(1,2,1)
sns.histplot(train['SalePrice'], kde=True, bins=30)
plt.yscale("log")
plt.title("SalePrice Distribution (log-scaled y-axis)")

# Boxplot (better view with log x-axis instead of y-axis)
plt.subplot(1,2,2)
sns.boxplot(x=np.log1p(train['SalePrice']))
plt.title("SalePrice (Log Transformed)")

plt.show()

print("Skewness of Sales Price :" ,skew(train['SalePrice']))

*Note : SalePrice has a skewness of 1.88, showing a strong right skew. This means most homes are moderately priced, with a few very expensive outliers stretching the distribution.*

In [ ]:
# Create a log-transformed target column (without replacing original)
train['SalePrice_log'] = np.log1p(train['SalePrice'])

# Quick check of distribution after log transform
plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
sns.histplot(train['SalePrice'], kde=True, bins=30)
plt.title("Original SalePrice")

plt.subplot(1,2,2)
sns.histplot(train['SalePrice_log'], kde=True, bins=30)
plt.title("Log-Transformed SalePrice")

plt.show()

*Note : I created a SalePrice_log feature to reduce skewness while keeping the original SalePrice intact. The log-transformed version looks closer to a normal distribution and will be useful for later modeling, while the raw prices remain for interpretability.*

**MSSubClass**

In [ ]:
train['MSSubClass'].value_counts()

In [ ]:
plt.figure(figsize=(10,4))
sns.countplot(x='MSSubClass', data=train)
plt.title("MSSubClass Distribution")
plt.xticks(rotation=45)
plt.show()

# Average SalePrice by MSSubClass
plt.figure(figsize=(10,4))
sns.barplot(x='MSSubClass', y='SalePrice', data=train, estimator=np.mean, ci=None)
plt.title("Average SalePrice by MSSubClass")
plt.xticks(rotation=45)
plt.show()

*Note : MSSubClass represents house type, and although it’s numeric, it behaves like a categorical variable. Some subclasses (like 20 and 60) dominate the dataset. Average sale price clearly varies across subclasses, showing that house type strongly influences price.*

In [ ]:
train['MSSubClass'] = train['MSSubClass'].astype(str)
test['MSSubClass'] = test['MSSubClass'].astype(str)

*Note : Since MSSubClass is a categorical code rather than a true numeric variable, we should treat it as a categorical feature. Converting it to string ensures models won’t misinterpret it as continuous. Some rare classes may later be grouped for stability.*

**MSZoning**

In [ ]:
train['MSZoning'].value_counts()

In [ ]:
# Countplot
plt.figure(figsize=(8,4))
sns.countplot(x='MSZoning', data=train)
plt.title("MSZoning Distribution")
plt.show()

# Average SalePrice by Zoning
plt.figure(figsize=(8,4))
sns.barplot(x='MSZoning', y='SalePrice', data=train, estimator=np.mean, ci=None)
plt.title("Average SalePrice by MSZoning")
plt.show()

*Note : Most houses fall under the "RL" (Residential Low Density) zoning category, followed by RM and FV. Sale prices vary by zoning: houses in “FV” and “RL” zones tend to fetch higher average prices, suggesting that zoning regulations have a direct impact on property value.*

In [ ]:
# Distribution
plt.figure(figsize=(8,4))
sns.histplot(train['LotFrontage'], bins=30, kde=True)
plt.title("LotFrontage Distribution")
plt.show()

# Relationship with SalePrice
plt.figure(figsize=(8,4))
sns.scatterplot(x='LotFrontage', y='SalePrice', data=train, alpha=0.6)
plt.title("LotFrontage vs SalePrice")
plt.show()

# Correlation
print("Correlation with SalePrice:", train['LotFrontage'].corr(train['SalePrice']))

*Note : LotFrontage is right-skewed, with many properties having smaller street frontage and fewer with very large frontage. Larger frontage is generally associated with higher prices, though the correlation is moderate. A few extreme values (outliers) are visible, which might need handling later during feature engineering.*

**LotArea**

In [ ]:
# Distribution
plt.figure(figsize=(8,4))
sns.histplot(train['LotArea'] ,bins = 30 ,kde= True)
plt.title("LotArea Distribution")
plt.show()

# Relationship with SalePrice
plt.figure(figsize=(8,4))
sns.scatterplot(x='LotArea' ,y= 'SalePrice' ,data = train ,alpha= 0.6 )
plt.title("LotArea vs SalesPrice")
plt.show()

# correlation
print("Correlation with SalesPrice :" ,train['LotArea'].corr(train['SalePrice']))

*Note : LotArea is highly right-skewed: most properties are clustered under 20,000 sq ft, with a few extremely large outliers stretching far to the right. While larger lots can increase price, the relationship is weak and noisy, suggesting lot size alone doesn’t strongly dictate sale price. Outliers may need capping or transformation during preprocessing.*

# BIVARAITE ANALYSIS

**Numerical Feature Scan**

In [ ]:
# select numerical columns
num_cols = train.select_dtypes(include=['int64', 'float64']).columns
print("Numerical columns:", list(num_cols))

In [ ]:
plt.figure(figsize=(12,8))
corr = train[num_cols].corr()

# Focus on SalePrice correlation
sns.heatmap(corr[['SalePrice']].sort_values(by='SalePrice', ascending=False),
            annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title("Correlation of Numerical Features with SalePrice")
plt.show()

In [ ]:
# visualizing top features

top_features = ['OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea', 'TotalBsmtSF']

plt.figure(figsize=(15,8))

for i, col in enumerate(top_features, 1):
    plt.subplot(2, 3, i)

    if train[col].nunique() < 15:
        sns.boxplot(x=col, y='SalePrice', data=train)
    else:
        sns.scatterplot(x=col, y='SalePrice', data=train, alpha=0.6)

    plt.title(f"{col} vs SalePrice")

plt.tight_layout()
plt.show()



OverallQual: Strongest driver — prices rise almost exponentially with quality.

GrLivArea: Bigger living area → higher price, but watch for outliers (very large homes priced lower than expected).

GarageCars: Prices increase with garage capacity, but maxes out around 3–4 cars.

GarageArea: Positive trend, but weaker than GarageCars.

TotalBsmtSF: Larger basements → higher price, though effect is weaker than living area.

**Boxplotss of SalePrice vs Category**

In [ ]:
top_cat_features = ['Neighborhood', 'ExterQual', 'KitchenQual', 'GarageType', 'FireplaceQu']

plt.figure(figsize=(15,12))

for i, col in enumerate(top_cat_features, 1):
    plt.subplot(3, 2, i)
    sns.boxplot(x=col, y='SalePrice', data=train)
    plt.xticks(rotation=45)
    plt.title(f"{col} vs SalePrice")

plt.tight_layout()
plt.show()

Neighborhood: Location heavily influences price — some neighborhoods consistently have higher prices.

ExterQual & KitchenQual: Higher quality materials/finish → higher sale prices.

GarageType: Presence and type of garage affects prices moderately.

FireplaceQu: Quality of fireplace adds value, though effect is less than overall quality or living area.

# Batch Summary for all Features

**Numerical cols**

In [ ]:
# Select numerical columns (excluding SalePrice and SalePrice_log)
num_cols = train.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_cols = [col for col in num_cols if col not in ['SalePrice' ,'SalePrice_log']]

# summary statistics
num_summary = train[num_cols].describe().T
num_summary['skewness'] = train[num_cols].skew()
num_summary['corr_with_saleprice'] = train[num_cols].corrwith(train['SalePrice'])
num_summary = num_summary.sort_values(by='corr_with_saleprice' ,ascending =False)

num_summary.head(15)

**Categorical Features**

In [ ]:
cat_cols = train.select_dtypes(include=['object']).columns

cat_summary_list = []

for col in cat_cols:
    num_categories = train[col].nunique()
    most_common = train[col].mode()[0]
    avg_saleprice = train.groupby(col)['SalePrice'].mean().sort_values(ascending=False).head(1).values[0]

    cat_summary_list.append({
        'Feature': col,
        'Num_Categories': num_categories,
        'Most_Common': most_common,
        'Avg_SalePrice': avg_saleprice
    })

# Convert list of dicts to DataFrame
cat_summary = pd.DataFrame(cat_summary_list)
cat_summary = cat_summary.sort_values(by='Avg_SalePrice', ascending=False)

cat_summary.head(15)

# Outlier Detection and Handling

In [ ]:
# Function for outlier detection
def detect_outliers(df , feature):
    Q1 = df[feature].quantile(0.25)
    Q3 = df[feature].quantile(0.75)

    #Inter-quartile range
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[feature] <lower ) | (df[feature] > upper)]
    print(f"{feature} : {outliers.shape[0]} outliers detected ")
    return outliers

# checks outliers for top numerical features
top_num_features = ['GrLivArea', 'LotArea', 'TotalBsmtSF', 'GarageArea', '1stFlrSF']

for col in top_num_features :
  detect_outliers(train ,col)

# Visual Inspection

In [ ]:
plt.figure(figsize=(15,5))
for i, col in enumerate(top_num_features, 1):
    plt.subplot(1,5,i)
    sns.boxplot(y=train[col])
    plt.title(col)
plt.tight_layout()
plt.show()

*Note : Outlier detection on key features shows some extremely large living areas and lot sizes. Most are valid observations, so instead of removing, we may cap or log-transform these variables later to reduce skew and stabilize model performance.*

**Feature Transformation**

In [ ]:
from scipy.stats import skew

# Select numerical columns excluding target
num_features = train.select_dtypes(include=['int64','float64']).drop(['SalePrice','SalePrice_log'], axis=1)

# Compute skewness
skewed_feats = num_features.apply(lambda x: skew(x.dropna())).sort_values(ascending=False)
# threshold for transformation
skewed_feats = skewed_feats[abs(skewed_feats) > 0.75]

print("Skewed features:", skewed_feats.index.tolist())


**Handle Skewed Numerical Features**

In [ ]:
# Apply log1p transformation to reduce skew
for col in skewed_feats.index:
    train[col] = np.log1p(train[col])
    test[col] = np.log1p(test[col])

**Encode categorical Features**

In [ ]:
# Separate target variable
y_train = train['SalePrice_log']  # or 'SalePrice'

# Drop target from train and test before encoding
X_train = train.drop(['SalePrice', 'SalePrice_log'], axis=1)
X_test = test.copy()  # drop target if present

In [ ]:
# One-hot encode categorical columns
X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)

# Align train/test features
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

In [ ]:
# Combine with target
train_processed = pd.concat([X_train, y_train], axis=1)

**Feature Scaling**

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

**Encode Categorical Features**

In [ ]:
# One-hot encoding for low-cardinality features
train = pd.get_dummies(train, columns=cat_cols, drop_first=True)
test = pd.get_dummies(test, columns=cat_cols, drop_first=True)

# Align train/test
train, test = train.align(test, join='left', axis=1, fill_value=0)

*Note : Skewed numerical features were log-transformed to reduce the effect of extreme values. All numerical features were scaled for uniformity. Categorical features were one-hot encoded to prepare for machine learning, ensuring consistent feature representation across train and test sets.*

# **EDA & Preprocessing Summary Report**

**1 .Missing Values**

Columns with high missing % were dropped (PoolQC, Alley, MiscFeature).

Moderate missing columns were filled (LotFrontage with 0 or median, Garage*, MasVnr*, etc.).

Remaining features have no missing values.

In [ ]:
missing_report = train.isnull().sum().sort_values(ascending=False)
print(missing_report[missing_report>0])

*Inference : Data completeness is ensured. Missing value handling followed industry-standard thresholds (drop >90% missing, impute others).*

**2. Target Variable**

SalePrice is right-skewed (skew = 1.88).

Log transformation applied (SalePrice_log) to stabilize variance.

In [ ]:
sns.histplot(train['SalePrice_log'], kde=True)
plt.title("Log-transformed SalePrice")
plt.show()

*Inference: Log-transformed target normalizes distribution, improving regression performance.*

**3 .Numerical Features**

Skewed features identified (GrLivArea, LotArea, TotalBsmtSF, etc.) and log-transformed.

Correlation heatmap identified top drivers: OverallQual, GrLivArea, GarageCars, GarageArea, TotalBsmtSF.

Outliers detected but handled via transformation instead of removal.

In [ ]:
# Get correlation with SalePrice and select top 20
corr_with_price = train.corr()['SalePrice'].sort_values(ascending=False).head(20)

plt.figure(figsize=(8,10))
sns.heatmap(corr_with_price.to_frame(), annot=True, cmap='coolwarm', fmt=".2f", cbar=False)
plt.title("Top 20 Features Correlated with SalePrice", fontsize=14)
plt.xticks(rotation=0)
plt.yticks(rotation=0)
plt.show()

*Inference: Top numerical drivers are prioritized for modeling. Extreme values mitigated via log-transform.*

**4 .Categorical Features**

Batch summary shows top categorical drivers: Neighborhood, ExterQual, KitchenQual, GarageType, FireplaceQu.

One-hot encoding applied for ML readiness.

In [ ]:
cat_summary.head(10)

*Inference: Categorical features encoded consistently. High-cardinality or rare categories handled appropriately.*

**5.Feature Scaling & Transformation**

Log-transform applied to skewed numerical features.

StandardScaler used to normalize feature scales.

Train and test aligned to ensure consistent feature space.

In [ ]:
# After preprocessing, feature space check
print("Train shape:", train.shape)
print("Test shape:", test.shape)

*Inference: All features are now normalized, scaled, and encoded, ready for machine learning.*